# RUD distortion GUI example

Detect wells on a uniformity/distortion (RUD) target, then plot local geometric
distortion. Well finding is automatic. Plots stay **inside the notebook**.

**How to run**

1. Run the cells from top to bottom.
2. In *Choose your images*, either keep the example folder or uncomment `IMAGE_DIR`.
3. The remaining cells show plots here. If saving is on, files go in a results folder.


## 1. Setup


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from qal import DistortionAnalyzer, DistortionVisualizer, RudDetector
from qal.data import rud_example_1, rud_image1_example_1


## 2. Choose your images

RUD distortion uses a **folder** of images that are all the same size.

Leave `IMAGE_DIR` as `None` to use QAL's built-in example.

To use your own images, uncomment the `IMAGE_DIR` line and paste a folder path.


In [ ]:
# Uncomment the next line and put the path to your image folder:
# IMAGE_DIR = "/path/to/your/rud_images"

# Leave this as None to use the built-in example images.
IMAGE_DIR = None


## 3. Saving plots and data

When `SAVE_RESULTS` is `True`:

- If you set `IMAGE_DIR`, files go next to that folder in `<folder_name>_results/`
- If you use the example images, files go next to this notebook in `rud_example_1_results/`


In [ ]:
# Set this to True to write plots and CSV files.
SAVE_RESULTS = True

# File names inside the results folder (used only when SAVE_RESULTS is True):
SAVE_DETECTION_PLOT = "detected_wells.png"
SAVE_DISTORTION_CURVE = "distortion_vs_image_height.png"
SAVE_DISTORTION_MAP = "distortion_map.png"
SAVE_DOTS_CSV = "detected_wells.csv"
SAVE_DISTORTION_CSV = "distortion.csv"

# Set True if the images do not cover the full field of view.
IGNORE_EXTRA = False


## 4. Load the image folder


In [ ]:
if IMAGE_DIR is None:
    image_dir = rud_example_1()
    results_dir = Path("rud_example_1_results")
    preview_image = rud_image1_example_1()
else:
    image_dir = str(Path(IMAGE_DIR).expanduser())
    image_folder = Path(image_dir)
    results_dir = image_folder.parent / f"{image_folder.name}_results"
    preview_image = None

if SAVE_RESULTS:
    results_dir.mkdir(parents=True, exist_ok=True)
    print(f"Results folder: {results_dir.resolve()}")
else:
    results_dir = None

print(f"Image folder: {image_dir}")

if preview_image is not None:
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(preview_image, cmap="gray")
    ax.set_title("Example image (first in folder)")
    ax.axis("off")
    plt.show()


## 5. Detect wells


In [ ]:
detector = RudDetector()
%matplotlib inline

detector.detect_dots_distortion(image_dir)
if detector.output is None:
    raise RuntimeError("No wells were found. Check the image folder and try again.")

n_images = len(detector.dots)
n_wells = sum(len(dots) for dots in detector.dots)
print(f"Found {n_wells} wells across {n_images} image(s).")


## 6. Review detected wells

This overlay stays in the notebook so you can check that the wells look right.


In [ ]:
first_image = detector.images[0]
first_dots = detector.dots[0]

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(first_image, cmap="gray")
ax.scatter(first_dots["x"], first_dots["y"], s=8, c="lime", alpha=0.7)
ax.set_title("Detected wells (first image)")
ax.axis("off")
plt.show()

if results_dir is not None:
    detection_path = results_dir / SAVE_DETECTION_PLOT
    fig.savefig(detection_path, bbox_inches="tight")
    print(f"Saved {detection_path}")


## 7. Compute distortion


In [ ]:
%matplotlib inline

analyzer = DistortionAnalyzer()
analyzer.compute_distortion(detector.output, ignore_extra=IGNORE_EXTRA)
distortion_df = pd.DataFrame(
    {
        "actual_distance_px": analyzer.output["Actual distances"],
        "distortion_percent": analyzer.output["Distortion"],
    }
)
distortion_df


## 8. Plot distortion


In [ ]:
visualizer = DistortionVisualizer()
visualizer.visualize_distortion(analyzer.output, save=False)

if results_dir is not None:
    figs = [plt.figure(number) for number in plt.get_fignums()]
    names = [SAVE_DISTORTION_CURVE, SAVE_DISTORTION_MAP]
    for fig, name in zip(figs, names):
        path = results_dir / name
        fig.savefig(path, bbox_inches="tight")
        print(f"Saved {path}")


## 9. Save tables as CSV


In [ ]:
if results_dir is None:
    print("SAVE_RESULTS is False, so no CSV files were written.")
else:
    dots_path = results_dir / SAVE_DOTS_CSV
    distortion_path = results_dir / SAVE_DISTORTION_CSV
    pd.concat(detector.dots, ignore_index=True).to_csv(dots_path, index=False)
    distortion_df.to_csv(distortion_path, index=False)
    print(f"Saved {dots_path}")
    print(f"Saved {distortion_path}")
